# CEOAI Practice 1 - Project KRAKEN Minimum Solution

Objective: create a first valid multimodal baseline:

1. Load slices, echoes, glyphs, and targets.
2. Convert each modality into cheap statistical features.
3. Train one simple model per subtask.
4. Export one submission CSV with all three subtask outputs.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_squared_error, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

In [ ]:
train_slices = np.load(DATA / "train_slices.npy")
test_slices = np.load(DATA / "test_slices.npy")
train_echoes = np.load(DATA / "train_echoes.npy")
test_echoes = np.load(DATA / "test_echoes.npy")
train_glyphs = pd.read_csv(DATA / "train_glyphs.csv")
test_glyphs = pd.read_csv(DATA / "test_glyphs.csv")
targets = pd.read_csv(DATA / "train_targets.csv")

print({
    "train_slices": train_slices.shape,
    "test_slices": test_slices.shape,
    "train_echoes": train_echoes.shape,
    "test_echoes": test_echoes.shape,
})

In [ ]:
def make_features(slices, echoes, glyph_df):
    image_features = np.concatenate([
        slices.mean(axis=(2, 3)),
        slices.std(axis=(2, 3)),
        slices.max(axis=(2, 3)),
    ], axis=1)
    echo_features = np.concatenate([
        echoes.mean(axis=1),
        echoes.std(axis=1),
        np.mean(np.isclose(echoes, 0), axis=1),
    ], axis=1)
    glyph_features = []
    for text in glyph_df["glyphs"]:
        tokens = text.split()
        glyph_features.append([
            len(tokens),
            len(set(tokens)),
            tokens.count("PHI"),
            sum(len(tok) for tok in tokens) / max(len(tokens), 1),
        ])
    return np.concatenate([image_features, echo_features, np.array(glyph_features, dtype=float)], axis=1)

X = make_features(train_slices, train_echoes, train_glyphs)
X_test = make_features(test_slices, test_echoes, test_glyphs)
coef_cols = [f"coef_{i}" for i in range(10)]
print(X.shape, X_test.shape)

In [ ]:
idx_train, idx_val = train_test_split(np.arange(len(X)), test_size=0.25, random_state=0, stratify=targets["topology_class"])
X_train, X_val = X[idx_train], X[idx_val]

coef_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
class_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
stability_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))

coef_model.fit(X_train, targets.loc[idx_train, coef_cols])
class_model.fit(X_train, targets.loc[idx_train, "topology_class"])
stability_model.fit(X_train, targets.loc[idx_train, "stability"])

coef_pred = coef_model.predict(X_val)
class_pred = class_model.predict(X_val)
stability_pred = np.clip(stability_model.predict(X_val), 0, 1)

print({
    "coef_mse": float(mean_squared_error(targets.loc[idx_val, coef_cols], coef_pred)),
    "class_macro_f1": float(f1_score(targets.loc[idx_val, "topology_class"], class_pred, average="macro")),
    "stability_rmse": float(np.sqrt(mean_squared_error(targets.loc[idx_val, "stability"], stability_pred))),
})

In [ ]:
test_coef = coef_model.predict(X_test)
test_class = class_model.predict(X_test)
test_stability = np.clip(stability_model.predict(X_test), 0, 1)

rows = []
for i, datapoint_id in enumerate(test_glyphs["datapointID"]):
    rows.append({
        "subtaskID": 1,
        "datapointID": datapoint_id,
        "answer": ";".join(f"{x:.6f}" for x in test_coef[i]),
    })
    rows.append({"subtaskID": 2, "datapointID": datapoint_id, "answer": int(test_class[i])})
    rows.append({"subtaskID": 3, "datapointID": datapoint_id, "answer": float(test_stability[i])})

submission = pd.DataFrame(rows)
submission.to_csv(OUT / "submission.csv", index=False)
assert len(submission) == 3 * len(test_glyphs)
print("wrote", OUT / "submission.csv")
submission.head(9)